# Hotel Customer No-Show Prediction — Exploratory Data Analysis

## Objective and problem framing

The objective is to predict whether a booking becomes a no-show so the hotel can target confirmation, deposit or reminder policies and reduce unused-room cost.

`no_show` is the target: `0` means the customer arrived and `1` means no-show. Because the target has two defined outcomes, this is **binary classification**. Missing a true no-show can leave inventory unused, so recall is important. False alarms can inconvenience reliable guests, so precision cannot be ignored. Model selection therefore uses **F2**, which weights recall twice as strongly as precision, with ROC-AUC and average precision as supporting metrics.

## 1. Setup and simple CSV loading

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from scipy.stats import chi2_contingency, mannwhitneyu

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

DATA_PATH = Path.cwd() / "data" / "data.csv"
df_raw = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
display(df_raw.head())

## 2. Combined schema and data-quality audit

In [ ]:
required_columns = {
    "booking_id", "no_show", "branch", "booking_month", "arrival_month",
    "arrival_day", "checkout_month", "checkout_day", "country", "first_time",
    "room", "price", "platform", "num_adults", "num_children",
}
target_numeric = pd.to_numeric(df_raw["no_show"], errors="coerce")
audit = pd.DataFrame({
    "check": ["Required schema", "Shape", "Duplicate rows", "Missing cells", "Columns with missing values", "Invalid target rows", "Unique booking IDs", "Negative checkout days"],
    "result": [
        "Pass" if required_columns.issubset(df_raw.columns) else "Fail",
        f"{df_raw.shape[0]:,} × {df_raw.shape[1]}",
        int(df_raw.duplicated().sum()), int(df_raw.isna().sum().sum()),
        int((df_raw.isna().sum() > 0).sum()), int((~target_numeric.isin([0, 1])).sum()),
        int(df_raw["booking_id"].nunique()), int((df_raw["checkout_day"] < 0).sum()),
    ],
})
display(audit)

column_audit = pd.DataFrame({
    "data_type": df_raw.dtypes.astype(str), "non_null": df_raw.notna().sum(),
    "missing": df_raw.isna().sum(), "missing_percent": df_raw.isna().mean() * 100,
    "unique_values": df_raw.nunique(dropna=True),
}).sort_values("missing_percent", ascending=False)
display(column_audit)

In [ ]:
missing = column_audit.loc[column_audit["missing"] > 0, ["missing", "missing_percent"]]
display(missing)
plt.figure(figsize=(9, 5))
sns.barplot(data=missing.reset_index(), x="missing_percent", y="index", color="#4C78A8")
plt.xlabel("Missing values (%)"); plt.ylabel("Feature"); plt.title("Missingness by feature")
plt.tight_layout(); plt.show()

The dataset has no exact duplicate rows and booking IDs are unique. One incomplete row has no valid target and must be removed. `room` and `price` contain material missingness, which will be imputed inside the training pipeline with missingness indicators. Negative checkout days are sign-format errors because all absolute values fall within 1–31.

## 3. Transparent cleaning for EDA

In [ ]:
df = df_raw.loc[df_raw["no_show"].isin([0, 1])].copy()
df["no_show"] = df["no_show"].astype(int)
text_columns = ["branch", "booking_month", "arrival_month", "checkout_month", "country", "first_time", "room", "platform"]
for column in text_columns:
    df[column] = df[column].astype(object).str.strip().str.title()

df["checkout_day"] = pd.to_numeric(df["checkout_day"], errors="coerce").abs()
df["num_adults"] = pd.to_numeric(df["num_adults"].replace({"one": 1, "two": 2, "One": 1, "Two": 2}), errors="coerce")
df["num_children"] = pd.to_numeric(df["num_children"], errors="coerce")
df["price_currency"] = df["price"].str.extract(r"^([A-Z]{3})\$")[0]
df["price_amount"] = pd.to_numeric(df["price"].str.replace(r"^[A-Z]{3}\$\s*", "", regex=True), errors="coerce")
print(f"Clean EDA dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

## 4. Descriptive statistics

In [ ]:
display(df[["no_show", "arrival_day", "checkout_day", "price_amount", "num_adults", "num_children"]].describe().T)
categorical_columns = ["branch", "booking_month", "arrival_month", "checkout_month", "country", "first_time", "room", "platform", "price_currency"]
categorical_summary = pd.DataFrame({
    "non_null": df[categorical_columns].notna().sum(),
    "missing": df[categorical_columns].isna().sum(),
    "categories": df[categorical_columns].nunique(),
    "most_frequent": [df[c].mode().iloc[0] for c in categorical_columns],
})
display(categorical_summary)

## 5. Univariate analysis

In [ ]:
target_counts = df["no_show"].value_counts().sort_index()
target_rates = df["no_show"].value_counts(normalize=True).sort_index()
display(pd.DataFrame({"count": target_counts, "proportion": target_rates}))
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="no_show", color="#4C78A8")
plt.title("Target distribution"); plt.xticks([0, 1], ["Show", "No-show"])
plt.tight_layout(); plt.show()

No-shows represent approximately **37.0%** of usable records. This is a meaningful minority class rather than an extreme imbalance. Stratified splitting and class-balanced models are still appropriate because accuracy alone could obscure missed no-shows.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for column, axis in zip(["branch", "country", "first_time", "room", "platform", "arrival_month"], axes.flatten()):
    order = df[column].value_counts().index
    sns.countplot(data=df, x=column, order=order, ax=axis, color="#59A14F")
    axis.set_title(column); axis.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df["price_amount"], bins=50, ax=axes[0], color="#4C78A8")
sns.countplot(data=df, x="num_adults", ax=axes[1], color="#59A14F")
sns.countplot(data=df, x="num_children", ax=axes[2], color="#F28E2B")
axes[0].set_title("Room price amount"); axes[1].set_title("Number of adults"); axes[2].set_title("Number of children")
plt.tight_layout(); plt.show()

## 6. Bivariate and multivariate analysis

In [ ]:
rate_features = ["branch", "country", "first_time", "room", "platform", "arrival_month", "price_currency"]
for column in rate_features:
    summary = df.groupby(column, dropna=False)["no_show"].agg(["count", "mean"]).sort_values("mean", ascending=False)
    summary = summary.rename(columns={"mean": "no_show_rate"})
    print()
    print(f"No-show rate by {column}")
    display(summary)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for column, axis in zip(["branch", "country", "first_time", "room", "platform", "arrival_month"], axes.flatten()):
    rates = df.groupby(column, dropna=False)["no_show"].mean().sort_values(ascending=False)
    sns.barplot(x=rates.index.astype(str), y=rates.values, ax=axis, color="#E15759")
    axis.set_title(f"No-show rate by {column}"); axis.set_ylabel("No-show rate"); axis.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
numeric_analysis = df[["arrival_day", "checkout_day", "price_amount", "num_adults", "num_children", "no_show"]]
plt.figure(figsize=(9, 7))
sns.heatmap(numeric_analysis.corr(method="spearman"), cmap="vlag", center=0, annot=True, fmt=".2f")
plt.title("Spearman correlation heatmap")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for column, axis in zip(["price_amount", "num_adults", "num_children"], axes):
    sns.boxplot(data=df, x="no_show", y=column, ax=axis, color="#9ECAE1", showfliers=False)
    axis.set_xticklabels(["Show", "No-show"]); axis.set_title(f"{column} by outcome")
plt.tight_layout(); plt.show()

Country is the strongest visible segmentation signal: customers from China have a substantially higher observed no-show rate. Changi also has a higher rate than Orchard. First-time customers, room type and arrival month show smaller differences. Platform and price currency show little separation. These are associations, not causal explanations or grounds for punitive customer treatment.

## 7. Statistical validation and effect sizes

In [ ]:
categorical_tests = []
for column in rate_features:
    table = pd.crosstab(df[column].fillna("Missing"), df["no_show"])
    chi2, p_value, _, _ = chi2_contingency(table)
    cramer_v = np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))
    categorical_tests.append({"feature": column, "test": "Chi-square", "p_value": p_value, "cramers_v": cramer_v})

categorical_tests = pd.DataFrame(categorical_tests).sort_values("cramers_v", ascending=False)
display(categorical_tests.style.format({"p_value": lambda x: "<1e-300" if x == 0 else f"{x:.2e}", "cramers_v": "{:.3f}"}))

In [ ]:
numeric_tests = []
for column in ["arrival_day", "checkout_day", "price_amount", "num_adults", "num_children"]:
    no_show_values = df.loc[df["no_show"] == 1, column].dropna()
    show_values = df.loc[df["no_show"] == 0, column].dropna()
    result = mannwhitneyu(no_show_values, show_values, alternative="two-sided")
    rank_biserial = 2 * result.statistic / (len(no_show_values) * len(show_values)) - 1
    numeric_tests.append({"feature": column, "p_value": result.pvalue, "rank_biserial": rank_biserial, "no_show_median": no_show_values.median(), "show_median": show_values.median()})

numeric_tests = pd.DataFrame(numeric_tests).sort_values("rank_biserial", key=abs, ascending=False)
display(numeric_tests.style.format({"p_value": "{:.2e}", "rank_biserial": "{:.3f}"}))

Country has the largest categorical effect (Cramér’s V ≈ 0.34), followed by branch (≈ 0.14). Other categorical effects are modest, and platform/currency provide little standalone separation. Numerical rank-biserial effects are small. No single variable is sufficient, supporting comparison of multivariate linear and nonlinear models.

## 8. EDA-to-modelling decisions

In [ ]:
decisions = pd.DataFrame([
    {"EDA finding": "Target is binary with 37% no-shows", "Decision": "Binary classification; stratified split and class balancing"},
    {"EDA finding": "Missed no-shows are costly but false alerts also matter", "Decision": "Primary metric F2; report recall, precision, PR-AUC and ROC-AUC"},
    {"EDA finding": "booking_id is unique", "Decision": "Remove identifier"},
    {"EDA finding": "One row has no valid target", "Decision": "Remove that row before splitting"},
    {"EDA finding": "Month capitalisation is inconsistent", "Decision": "Standardise category case"},
    {"EDA finding": "Adults appear as words and digits", "Decision": "Map words and convert to numeric"},
    {"EDA finding": "Checkout days include negative signs", "Decision": "Use absolute value after domain validation"},
    {"EDA finding": "Prices contain SGD/USD prefixes", "Decision": "Separate amount and currency; let nonlinear model capture interaction"},
    {"EDA finding": "Room and price contain substantial missingness", "Decision": "Pipeline imputation with missingness indicators"},
    {"EDA finding": "Timing and party size may combine nonlinearly", "Decision": "Engineer lead months, stay length and total guests"},
    {"EDA finding": "Country/branch strongest; other effects weak", "Decision": "Compare Logistic Regression, Random Forest and Gradient Boosting"},
])
display(decisions)

## 9. Conclusion

The dataset supports leakage-safe binary classification after transparent cleaning. The pipeline will prioritize F2 and enforce a minimum precision guardrail when tuning the classification threshold. This aligns the model with reducing missed no-shows without flagging nearly every booking.